# Construction du dataset — Segmentation YOLO

**Projet** : Gallica Images — Classification gravures sur bois vs cuivre  
**Date**   : Avril 2026

Ce notebook construit le dataset d'images segmentées pour entraîner
le classifieur bois/cuivre. Il télécharge les pages depuis les sources
IIIF et l'API BnF, puis extrait les illustrations avec YOLO.

---

## Sources bois

| Dossier | ARK / Source | Lieu & Date | Graveur |
|---|---|---|---|
| `bois_salomon_rouille_lyon1557` | `btv1b2200047r` — API BnF | Lyon, 1557 | Bernard Salomon |
| `bois_wickram_behem_mayence1545` | `bsb10139926` — BSB Munich IIIF | Mayence, 1545 | Jörg Wickram (1505?–1560?) |
| `bois_solis_feyerabend_francfort1581` | `bsb00087854` — BSB Munich IIIF | Francfort-sur-le-Main, 1581 | Virgil Solis |

## Sources cuivre

| Dossier | ARK / Source | Lieu & Date | Graveur |
|---|---|---|---|
| `cuivre_savery_farnaby_paris1637` | `bsb10863401` — BSB Munich IIIF | Paris, 1637 | Francisco Clein (inv.) & Salomon Savery (sculp.) |
| `cuivre_passe_metamorphoseon` | `bpt6k15218623` — PDF Gallica | — | Crispin de Passe |
| `cuivre_passe_nasonis` | `bpt6k1522448r` — PDF Gallica | — | Crispin de Passe |
| `cuivre_renouard_traduites` | `bpt6k6277348n` — PDF Gallica | — | non renseigné |
| `cuivre_renouard_traduittes` | `bpt6k722055` — PDF Gallica | — | non renseigné |

Ces 8 éditions sont celles téléchargées et segmentées **dans ce notebook**
(sections 3-4 ci-dessous). Le corpus complet utilisé pour le split
(27 éditions, section 7-8) s'est enrichi depuis avec des éditions
supplémentaires segmentées séparément et déjà présentes sur le disque.

---

## Split train / val / test (v4)

Le split ne se fait plus par tirage aléatoire d'images individuelles
(voir section 8) : chaque édition est assignée en entier à un seul split,
pour éviter qu'une même édition n'apparaisse à la fois en train et en test.
L'augmentation (flip, rotation, jitter de couleur) n'est plus appliquée ici
par génération de fichiers physiques — elle est faite à la volée pendant
l'entraînement via `gallica_utils.TRANSFORM_TRAIN` (voir `02_entrainement.ipynb`).

---

⚠️ **Note mémoire GPU** : YOLO est chargé pour la segmentation puis libéré
avant le split du dataset. Ne pas charger ResNet50 dans ce notebook.

---

## 1. Configuration

In [ ]:
import sys
sys.path.insert(0, "..")
from gallica_utils import (
    charger_yolo, segmenter_page, segmenter_corpus,
    liberer_yolo, telecharger_pages_iiif,
    stats_illustrations,
    BASE_URL, ARK_SALOMON
)

import os
import shutil
import requests
import subprocess
import torch
from PIL import Image
from io import BytesIO

# ── Chemins ─────────────────────────────────────────────────
DOSSIER_IMAGES_BRUTES = "../../data/editions_ovide/sources"
DOSSIER_SEGMENTEES    = "../../data/editions_ovide/segmentees"
DOSSIER_DATASET       = "../../data/editions_ovide/datasets"
DOSSIER_PDF           = "../../data/editions_ovide/sources/cuivre_pdfs_bruts"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

## 2. Chargement de YOLOv5

Modèle `seglinglin/Historical-Illustration-Extraction` depuis Hugging Face.  
Entraîné sur des documents historiques — détecte les illustrations dans les pages imprimées.

In [16]:
# Cloner YOLOv5 si absent
if not os.path.exists("../../../yolov5_repo"):
    subprocess.run(["git", "clone",
                    "https://github.com/ultralytics/yolov5.git",
                    "../../../yolov5_repo"])
    print("✓ YOLOv5 cloné")

modele_yolo = charger_yolo(yolov5_repo="../../../yolov5_repo")

✓ YOLO chargé — classes : {0: 'illustration'}


## 3. Classe bois

### 3.1 Bernard Salomon — Lyon 1557 — API BnF

##### <u>Titre :</u> Excellente figueren ghesneden vuyten vppersten Poëte Ouidius vuyt vyfthien boucken der veranderinghe met huerlier bedietsele. Duer Guilliaume Borluit burgher der stede van Ghendt.

##### <u>Graveur :</u> Salomon, Bernard

##### <u>ARK :</u> `btv1b2200047r` — Source : API BnF (`/api/ouvrages/{ark}/illustrations`)

##### <u>Dossier :</u> `bois_salomon_rouille_lyon1557/`

In [17]:
r             = requests.get(f"{BASE_URL}/api/ouvrages/{ARK_SALOMON}/illustrations", timeout=30)
illustrations = r.json()
valides       = [
    illus for illus in illustrations
    if illus.get("metas", {}).get("content_embedding")
    and len(illus["metas"]["content_embedding"]) == 768
]
print(f"Illustrations Salomon avec embedding : {len(valides)}")

Illustrations Salomon avec embedding : 184


In [18]:
dossier_brut_salomon = f"{DOSSIER_IMAGES_BRUTES}/bois_salomon_rouille_lyon1557"
os.makedirs(dossier_brut_salomon, exist_ok=True)
pages_bois_brutes = []

for i, illus in enumerate(valides):
    print(f"  {i+1}/{len(valides)}...", end="\r")
    metas  = illus.get("metas", {})
    url    = metas.get("link", "")
    view   = illus.get("view_number", i)
    chemin = f"{dossier_brut_salomon}/salomon_f{view:03d}.jpg"

    if os.path.exists(chemin):
        pages_bois_brutes.append(chemin)
        continue
    try:
        r   = requests.get(url, timeout=15)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.save(chemin)
        pages_bois_brutes.append(chemin)
    except Exception as e:
        print(f"\n  Erreur page {view} : {e}")

print(f"\n✓ {len(pages_bois_brutes)} pages Salomon sauvegardées")

  184/184...
✓ 184 pages Salomon sauvegardées


In [19]:
segmenter_corpus(
    pages_bois_brutes,
    f"{DOSSIER_SEGMENTEES}/bois_salomon_rouille_lyon1557",
    modele_yolo,
    conf_thres=0.25
)

  184/184...
✓ 204 illustrations extraites dans ../../data/bois_cuivre/segmentees/bois_salomon_rouille_lyon1557


204

### 3.2 Jörg Wickram — Mayence 1545 — BSB Munich `bsb10139926`

##### <u>Titre :</u> P. Ouidij Nasonis dess aller sinnreichsten Poeten Metamorphosis / Das ist : von der wunderbarlicher Verenderung der Gestalten der Menschen / Thiern vnd anderer Creaturen etc. Jedermann lüstlich / besonders aber allen Malern / Bildthauwern / vnnd dergleichen Kunstlern nützlich zu gebrauchen.

##### <u>Graveur :</u> Wickram, Jörg (1505?–1560?)

##### <u>ARK :</u> `bsb10139926` — Source : BSB Munich IIIF

##### <u>Dossier :</u> `bois_wickram_behem_mayence1545/`

In [20]:
pages_mayence = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb10139926/manifest",
    f"{DOSSIER_IMAGES_BRUTES}/bois_wickram_behem_mayence1545",
    prefixe="wickram"
)

segmenter_corpus(
    pages_mayence,
    f"{DOSSIER_SEGMENTEES}/bois_wickram_behem_mayence1545",
    modele_yolo,
    conf_thres=0.25
)

Pages trouvées : 332 — Ovidius Naso, Publius: P. Ouidij Nasonis deß aller sinnreich
  332/332...
✓ 332 pages sauvegardées dans ../../data/bois_cuivre/sources/bois_wickram_behem_mayence1545
  332/332...
✓ 70 illustrations extraites dans ../../data/bois_cuivre/segmentees/bois_wickram_behem_mayence1545


70

### 3.3 Virgil Solis — Francfort-sur-le-Main 1581 — BSB `bsb00087854`

##### <u>Titre :</u> P. Ovidii Metamorphosis, Oder : Wunderbarliche vnnd seltzame Beschreibung / von der Menschen / Thiern / vnnd anderer Creaturen Veränderung auch von dem Wandeln / Leben vnd Thaten der Götter / Martis / Veneris / Mercurij / etc. Allen Poeten / Malern / Goldschmiden / Bildthauwern / vnnd Liebhabern der edlen Poesi vnd fürnembsten Künsten / Nützlich vnd lustig zu lesen / Jetzt widerum auff ein newes / dem gemeinen Vatterlande Teutscher Sprach zu grossem nutz vnd dienst auss sonderlichem fleiss mit schönen Figurn renouiert / corrigiert / vnd an Tag geben / durch Sigmund Feyerabendt Buchhändlern.

##### <u>Graveur :</u> Solis, Virgil

##### <u>ARK :</u> `bsb00087854` — Source : BSB Munich IIIF

##### <u>Dossier :</u> `bois_solis_feyerabend_francfort1581/`

In [21]:
pages_bsb87854 = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb00087854/manifest",
    f"{DOSSIER_IMAGES_BRUTES}/bois_solis_feyerabend_francfort1581",
    prefixe="solis"
)

segmenter_corpus(
    pages_bsb87854,
    f"{DOSSIER_SEGMENTEES}/bois_solis_feyerabend_francfort1581",
    modele_yolo,
    conf_thres=0.25
)

Pages trouvées : 435 — Ovidius Naso, Publius: P. Ovidii Metamorphosis, Oder: Wunder
  435/435...
✓ 435 pages sauvegardées dans ../../data/bois_cuivre/sources/bois_solis_feyerabend_francfort1581
  435/435...
✓ 199 illustrations extraites dans ../../data/bois_cuivre/segmentees/bois_solis_feyerabend_francfort1581


199

## 4. Classe cuivre

### 4.1 Clein & Savery — Paris 1637 — BSB `bsb10863401`

##### <u>Titre :</u> Pub. Ovidii Nasonis Metamorphoseon libri XV, ad fidem editionum optimarum et codicum manuscriptorum examinati, animadversi, necnon notis illustrati, opera et studio Thomae Farnabii. Editio nunc primum in Gallia et multis figuris aeneis adornata.

##### <u>Graveur :</u> Clein, Francisco (inv.) et Savery, Salomon (sculp.)

##### <u>ARK :</u> `bsb10863401` — Source : BSB Munich IIIF

##### <u>Dossier :</u> `cuivre_savery_farnaby_paris1637/`

In [22]:
pages_munich = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb10863401/manifest",
    f"{DOSSIER_IMAGES_BRUTES}/cuivre_savery_farnaby_paris1637",
    prefixe="clein"
)

segmenter_corpus(
    pages_munich,
    f"{DOSSIER_SEGMENTEES}/cuivre_savery_farnaby_paris1637",
    modele_yolo,
    conf_thres=0.25
)

Pages trouvées : 138 — Ovidius Naso, Publius: Metamorphoseon libri XV.
  138/138...
✓ 138 pages sauvegardées dans ../../data/bois_cuivre/sources/cuivre_savery_farnaby_paris1637
  138/138...
✓ 25 illustrations extraites dans ../../data/bois_cuivre/segmentees/cuivre_savery_farnaby_paris1637


25

### 4.2 PDFs Gallica — 4 éditions cuivre

| Dossier | ARK | Titre abrégé | Graveur |
|---|---|---|---|
| `cuivre_passe_metamorphoseon` | `bpt6k15218623` | Metamorphoseon Ovidianarum typi aliquot... | Crispin de Passe |
| `cuivre_passe_nasonis` | `bpt6k1522448r` | P. Ovid. Nasonis XV Metamorphoseon librorum figurae... | Crispin de Passe |
| `cuivre_renouard_traduites` | `bpt6k6277348n` | Les Métamorphoses d'Ovide, traduites en prose françoise... | non renseigné |
| `cuivre_renouard_traduittes` | `bpt6k722055` | Les métamorphoses d'Ovide, traduittes en prose françoise... | non renseigné |

In [23]:
import fitz  # PyMuPDF — pip install pymupdf

# Correspondance nom de fichier PDF → nom de dossier lisible
NOM_DOSSIERS_PDF = {
    "bpt6k15218623" : "cuivre_passe_metamorphoseon",
    "bpt6k1522448r" : "cuivre_passe_nasonis",
    "bpt6k6277348n" : "cuivre_renouard_traduites",
    "bpt6k722055"   : "cuivre_renouard_traduittes",
}

def extraire_pages_pdf(chemin_pdf, dossier_sortie, dpi=150):
    """Convertit chaque page d'un PDF en JPG."""
    os.makedirs(dossier_sortie, exist_ok=True)
    doc   = fitz.open(chemin_pdf)
    pages = []
    for i, page in enumerate(doc):
        mat    = fitz.Matrix(dpi/72, dpi/72)
        pix    = page.get_pixmap(matrix=mat)
        chemin = f"{dossier_sortie}/page{i+1:03d}.jpg"
        pix.save(chemin)
        pages.append(chemin)
        print(f"  {i+1}/{len(doc)}...", end="\r")
    print(f"\n✓ {len(pages)} pages extraites depuis {os.path.basename(chemin_pdf)}")
    return pages

def trouver_nom_dossier(nom_fichier):
    """Trouve le nom de dossier lisible depuis le nom du fichier PDF."""
    for ark, nom in NOM_DOSSIERS_PDF.items():
        if ark in nom_fichier:
            return nom
    return os.path.splitext(nom_fichier)[0][:50]

# Traiter les 4 PDFs
toutes_pages_pdf = []
for fichier in sorted(os.listdir(DOSSIER_PDF)):
    if fichier.endswith(".pdf"):
        nom_dossier = trouver_nom_dossier(fichier)
        dossier     = f"{DOSSIER_IMAGES_BRUTES}/cuivre_pdf/{nom_dossier}"
        print(f"\nTraitement : {fichier[:60]}")
        print(f"  → dossier : {nom_dossier}")
        pages = extraire_pages_pdf(f"{DOSSIER_PDF}/{fichier}", dossier)
        toutes_pages_pdf.append((nom_dossier, pages))

print(f"\n✓ Total pages PDF : {sum(len(p) for _, p in toutes_pages_pdf)}")


Traitement : Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide_(0043_bpt
  → dossier : cuivre_renouard_traduittes
  746/746...
✓ 746 pages extraites depuis Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide_(0043_bpt6k722055.pdf

Traitement : Metamorphoseon_Ovidianarum_typi_aliquot_artificiosissimè_[..
  → dossier : cuivre_passe_metamorphoseon
  313/313...
✓ 313 pages extraites depuis Metamorphoseon_Ovidianarum_typi_aliquot_artificiosissimè_[...]Ovide_(0043_bpt6k15218623.pdf

Traitement : P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wilhelm_bpt6k
  → dossier : cuivre_passe_nasonis
  284/284...
✓ 284 pages extraites depuis P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wilhelm_bpt6k1522448r.pdf

Traitement : [Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide_(0043_bpt
  → dossier : cuivre_renouard_traduites
  1226/1226...
✓ 1226 pages extraites depuis [Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide_(0043_bpt6k6277348n.pdf

✓ Total pages PDF : 2569


In [24]:
# Segmenter chaque édition dans son propre sous-dossier
for nom_dossier, pages in toutes_pages_pdf:
    dossier_sortie = f"{DOSSIER_SEGMENTEES}/cuivre_pdf/{nom_dossier}"
    print(f"\n{nom_dossier}")
    segmenter_corpus(pages, dossier_sortie, modele_yolo, conf_thres=0.25)


cuivre_renouard_traduittes
  746/746...
✓ 30 illustrations extraites dans ../../data/bois_cuivre/segmentees/cuivre_pdf/cuivre_renouard_traduittes

cuivre_passe_metamorphoseon
  313/313...
✓ 143 illustrations extraites dans ../../data/bois_cuivre/segmentees/cuivre_pdf/cuivre_passe_metamorphoseon

cuivre_passe_nasonis
  284/284...
✓ 148 illustrations extraites dans ../../data/bois_cuivre/segmentees/cuivre_pdf/cuivre_passe_nasonis

cuivre_renouard_traduites
  1226/1226...
✓ 36 illustrations extraites dans ../../data/bois_cuivre/segmentees/cuivre_pdf/cuivre_renouard_traduites


## 5. Libérer la mémoire GPU

⚠️ Obligatoire avant de lancer le fine-tuning ResNet50 dans le notebook 05.

In [25]:
modele_yolo = liberer_yolo(modele_yolo)

✓ Mémoire GPU libérée


## 6. Récapitulatif des illustrations segmentées


Nettoyage des données à la main à cette étape;

In [2]:
stats_illustrations(DOSSIER_SEGMENTEES)

Illustrations segmentées :

── BOIS ──
  bois_eskrich_rouille_lyon1556                      : 42
  bois_leroy_gueynard_lyon1510                       : 19
  bois_salomon_rouille_lyon1557                      : 161
  bois_solis_feyerabend_francfort1581                : 184
  bois_wickram_behem_mayence1545                     : 50

── CUIVRE ──
  cuivre_baur_sn_augsbourg1709                       : 161
  cuivre_baur_sn_vienne1639                          : 125
  cuivre_blanchin_berthelin_rouen1651                : 17
  cuivre_borcht_plantin_anvers1591                   : 182
  cuivre_bouche_blaeu_amsterdam1702                  : 126
  cuivre_briot_drobet_lyon1628                       : 28
  cuivre_depasse_depasse_koln1602                    : 134
  cuivre_depasse_jansonius_arnhem1607                : 136
  cuivre_franco_giunta_venise1584                    : 15
  cuivre_gaultier_guillemot_paris1610                : 16
  cuivre_gaultier_sn_paris1616                       : 14
  cuivre_ga

## 7. Corpus retenu pour le split (v4)

Le récapitulatif ci-dessus ne couvre que les 8 éditions téléchargées dans ce
notebook. Le corpus utilisé pour l'entraînement s'est enrichi au fil du projet
jusqu'à **27 éditions candidates** (5 bois, 22 cuivre), toutes déjà présentes
sous `data/editions_ovide/segmentees/`. `SOURCES` ci-dessous fixe explicitement
la liste **effectivement retenue** — **5 bois, 17 cuivre** — c'est le point de
référence pour savoir ce qui entre dans le dataset et ce qui en est
volontairement exclu.

**Cas particuliers traités avant le split :**

- **`cuivre_ht_molin_lyon1697_t4/t5/t6`** — trois tomes d'une même édition,
  fusionnés en un seul dossier `cuivre_ht_molin_lyon1697/` (17 images). Les
  garder séparés aurait compté un seul livre comme trois « éditions »
  indépendantes, ce qui aurait autorisé des images du même livre à se
  retrouver dans des splits différents.
- **`cuivre_goltzius_goltzius_haarlem1589_couleur`** — **exclue** de `SOURCES`.
  Mêmes noms de fichiers, mêmes boîtes de détection, même effectif (38/38)
  que `cuivre_goltzius_goltzius_haarlem1589` — c'est la même gravure
  numérisée deux fois (une fois en couleur). Reste disponible sur le disque
  pour un contrôle de généralisation a posteriori, mais ne participe pas au
  split.
- **`baur`, `depasse`, `gaultier`, `tempesta`** — chacun de ces graveurs a
  plusieurs dossiers, correspondant à des éditions différentes mais
  susceptibles de réutiliser les mêmes plaques de cuivre d'un tirage à
  l'autre (pratique courante : plaques revendues ou héritées par un autre
  imprimeur). Plutôt que de les regrouper, **un seul exemplaire par graveur
  est conservé** — le plus ancien, considéré comme l'édition d'origine :
  - `baur` : `vienne1639` conservé, `augsbourg1709` exclu
  - `depasse` : `koln1602` conservé, `arnhem1607` exclu
  - `gaultier` : `guillemot1610` conservé, `sn1616` et `veuveguillemot1614` exclus
  - `tempesta` : `anvers1606` conservé, `amsterdam1610` exclu

  Cette hypothèse (mêmes plaques réutilisées) n'a été vérifiée avec certitude
  que pour `baur` et `depasse` (confirmé) — pour `gaultier` et `tempesta`,
  c'est une précaution par analogie, pas un fait établi.

Les autres cas de graveurs présents dans plusieurs dossiers n'existent pas
au-delà de ces quatre — toutes les autres éditions ont un graveur unique.

Un autre cas a été identifié mais **volontairement laissé tel quel** :
`bois_solis_feyerabend_francfort1581` est une copie en miroir de
`bois_salomon_rouille_lyon1557` (pratique courante de copie sans repasser
par le dessin original). Les deux sont actuellement dans le même split
(train) par la façon dont l'algorithme du split (section 8) les a triés —
sans contrainte de code forcée, par choix.

In [5]:
SOURCES = {
    "bois": [
        "bois_eskrich_rouille_lyon1556",
        "bois_leroy_gueynard_lyon1510",
        "bois_salomon_rouille_lyon1557",
        "bois_solis_feyerabend_francfort1581",
        "bois_wickram_behem_mayence1545",
    ],
    "cuivre": [
        "cuivre_baur_sn_vienne1639",
        "cuivre_blanchin_berthelin_rouen1651",
        "cuivre_borcht_plantin_anvers1591",
        "cuivre_bouche_blaeu_amsterdam1702",
        "cuivre_briot_drobet_lyon1628",
        "cuivre_depasse_depasse_koln1602",
        "cuivre_franco_giunta_venise1584",
        "cuivre_gaultier_guillemot_paris1610",
        "cuivre_goltzius_goltzius_haarlem1589",
        "cuivre_ht_molin_lyon1697",
        "cuivre_isaac_langelier_paris1617",
        "cuivre_mathieu_langelier_paris1619",
        "cuivre_monconet_sommaville_paris1660",
        "cuivre_philippe_hackiana_leyde1670",
        "cuivre_savery_farnaby_paris1637",
        "cuivre_tempesta_dejode_anvers1606",
        "cuivre_weyen_barbin_paris1669",
    ],
}

print("Corpus retenu :\n")
for classe, dossiers in SOURCES.items():
    total = sum(
        len([f for f in os.listdir(f"{DOSSIER_SEGMENTEES}/{d}") if f.endswith(".jpg")])
        for d in dossiers if os.path.exists(f"{DOSSIER_SEGMENTEES}/{d}")
    )
    print(f"  {classe:8s} : {len(dossiers)} éditions — {total} illustrations")

Corpus retenu :

  bois     : 5 éditions — 456 illustrations
  cuivre   : 17 éditions — 1173 illustrations


## 8. Split train / val / test — groupé par édition

**Le problème corrigé ici** : dans `03_entrainement_v3.ipynb`, le split se
faisait en mélangeant toutes les illustrations d'une classe (toutes éditions
confondues) puis en tirant au sort au niveau de l'image. Deux illustrations
du même livre — même papier, même graveur, même bruit de numérisation —
pouvaient donc atterrir l'une en train, l'autre en test. Le modèle pouvait
apprendre à reconnaître le style d'une édition plutôt que la technique de
gravure, sans que le test set (contaminé par le même style) ne puisse le
détecter.

**Le principe ici** : chaque édition est assignée **en entier** à un seul
split. Aucune image d'une édition de test n'a de « cousine » vue à
l'entraînement — le score obtenu mesurera une vraie généralisation.

**La difficulté** : les éditions ont des tailles très inégales (3 à 184
images) et, côté bois, il n'y en a que 5. Un tirage aléatoire pur pourrait
placer une grosse édition entière dans le test et déséquilibrer complètement
les proportions. L'algorithme ci-dessous trie les éditions par taille
décroissante (ordre aléatoirisé au préalable pour départager les ex-aequo)
et assigne chacune au split qui en a le plus besoin par rapport à sa cible
(70 / 15 / 15 %) — une répartition gloutonne qui reste proche des proportions
cibles malgré la contrainte de ne jamais couper une édition.

La composition exacte (quelle édition va où) est **affichée et sauvegardée**
dans `resultats/evaluation_modeles/bois_cuivre/split_v4_editions.json` pour
que le split soit traçable et reproductible (seed fixe).

In [6]:
import random
import json

SEED = 42
RATIOS = (0.70, 0.15, 0.15)
DOSSIER_RESULTATS_EVAL = "../../resultats/evaluation_modeles/bois_cuivre"


def repartir_editions_par_groupe(dossiers, dossier_seg, ratios=RATIOS, seed=SEED):
    """
    Répartit des éditions entières (jamais coupées) entre train/val/test.

    Algorithme glouton : les éditions sont triées par taille décroissante
    (ordre aléatoirisé au préalable pour départager les ex-aequo), puis
    chacune est assignée au split qui en a le plus besoin par rapport à
    sa cible (ratios). Retourne (repartition, comptes) où
    repartition[split] = liste des éditions assignées à ce split.
    """
    rng = random.Random(seed)
    infos = []
    for d in dossiers:
        chemin = f"{dossier_seg}/{d}"
        if not os.path.exists(chemin):
            print(f"  ⚠ dossier introuvable, ignoré : {d}")
            continue
        n = len([f for f in os.listdir(chemin) if f.endswith(".jpg")])
        if n > 0:
            infos.append((d, n))

    rng.shuffle(infos)
    infos.sort(key=lambda x: x[1], reverse=True)

    total = sum(n for _, n in infos)
    cibles = {"train": ratios[0] * total, "val": ratios[1] * total, "test": ratios[2] * total}
    comptes = {"train": 0, "val": 0, "test": 0}
    repartition = {"train": [], "val": [], "test": []}

    for dossier, n in infos:
        split = max(comptes, key=lambda s: cibles[s] - comptes[s])
        repartition[split].append(dossier)
        comptes[split] += n

    return repartition, comptes


# Nettoyer l'ancien dataset
for split in ["train", "val", "test"]:
    for classe in ["bois", "cuivre"]:
        dossier = f"{DOSSIER_DATASET}/{split}/{classe}"
        if os.path.exists(dossier):
            shutil.rmtree(dossier)
        os.makedirs(dossier, exist_ok=True)

trace = {}

for classe, dossiers in SOURCES.items():
    repartition, comptes = repartir_editions_par_groupe(dossiers, DOSSIER_SEGMENTEES)
    trace[classe] = repartition
    total_classe = sum(comptes.values())

    print(f"\n{'='*70}\nClasse : {classe}  ({total_classe} images, {len(dossiers)} éditions)\n{'='*70}")
    for split in ["train", "val", "test"]:
        pct = 100 * comptes[split] / total_classe
        print(f"\n  {split.upper()} — {comptes[split]} images ({pct:.1f}%) :")
        for edition in repartition[split]:
            n = len([f for f in os.listdir(f"{DOSSIER_SEGMENTEES}/{edition}") if f.endswith(".jpg")])
            print(f"    - {edition:45s} ({n} images)")

    for split, editions in repartition.items():
        for edition in editions:
            chemin_edition = f"{DOSSIER_SEGMENTEES}/{edition}"
            for f in os.listdir(chemin_edition):
                if f.endswith(".jpg"):
                    shutil.copy(f"{chemin_edition}/{f}",
                                f"{DOSSIER_DATASET}/{split}/{classe}/{edition}_{f}")

# Trace de la répartition — reproductible et consultable indépendamment du notebook
os.makedirs(DOSSIER_RESULTATS_EVAL, exist_ok=True)
with open(f"{DOSSIER_RESULTATS_EVAL}/split_v4_editions.json", "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "ratios_cibles": RATIOS, "repartition": trace}, f, indent=2, ensure_ascii=False)

print(f"\n{'='*70}\n✓ Dataset final (v4 — split groupé par édition) :\n{'='*70}")
for split in ["train", "val", "test"]:
    nb_b = len([f for f in os.listdir(f"{DOSSIER_DATASET}/{split}/bois")   if f.endswith(".jpg")])
    nb_c = len([f for f in os.listdir(f"{DOSSIER_DATASET}/{split}/cuivre") if f.endswith(".jpg")])
    print(f"  {split:5s} : {nb_b:4d} bois + {nb_c:4d} cuivre = {nb_b + nb_c} total")
print(f"\n✓ Trace de la répartition sauvegardée : {DOSSIER_RESULTATS_EVAL}/split_v4_editions.json")


Classe : bois  (456 images, 5 éditions)

  TRAIN — 345 images (75.7%) :
    - bois_solis_feyerabend_francfort1581           (184 images)
    - bois_salomon_rouille_lyon1557                 (161 images)

  VAL — 50 images (11.0%) :
    - bois_wickram_behem_mayence1545                (50 images)

  TEST — 61 images (13.4%) :
    - bois_eskrich_rouille_lyon1556                 (42 images)
    - bois_leroy_gueynard_lyon1510                  (19 images)

Classe : cuivre  (1173 images, 17 éditions)

  TRAIN — 824 images (70.2%) :
    - cuivre_borcht_plantin_anvers1591              (182 images)
    - cuivre_monconet_sommaville_paris1660          (148 images)
    - cuivre_tempesta_dejode_anvers1606             (139 images)
    - cuivre_mathieu_langelier_paris1619            (135 images)
    - cuivre_depasse_depasse_koln1602               (134 images)
    - cuivre_goltzius_goltzius_haarlem1589          (38 images)
    - cuivre_ht_molin_lyon1697                      (17 images)
    - cuivre_gau